# S7 — 오염 시행 제거 효과: 피험자 단위 5-fold 교차검증

VS Code에서 이 노트북을 열고 **Google Colab GPU 런타임**을 커널로 선택하세요. 새 Google 계정의 `MyDrive/BCI_Research`를 사용합니다. 준비 셀 1–3을 먼저 실행한 다음 학습 셀을 원하는 순서로 실행합니다. **현재 노트북에는 학습 결과가 없으며 셀을 실행하기 전에는 재학습하지 않습니다.**

49명의 평가 가능 피험자를 고정된 5개 테스트 그룹으로 나눕니다. 각 그룹에서 해당 테스트 피험자들은 학습과 조기종료 검증에 사용되지 않습니다. `dirty_train`과 `clean_train`은 같은 깨끗한 테스트 시행에서 비교합니다. 5-fold × 3모델 × 2조건 × 3시드 = **90회 학습**이고, 각 조합은 테스트 피험자 49명의 점수를 저장합니다. 완료된 fold는 재실행할 때 건너뜁니다.

결과는 `results/rigorous_group5`에 저장됩니다. 기존 LOSO 결과와 직접 합치거나 점수 차이를 방법 한 가지의 효과로 해석하지 마세요. 신뢰구간은 모델을 공유하는 피험자들을 한 묶음으로 다루는 fold 단위 재표본추출을 사용하지만, fold가 5개라 탐색적입니다.

**전처리 한계:** 저장된 HDF5에서 오염 시행을 배제하고 깨끗한 시행으로 EEG 정규화를 다시 계산합니다. 원본 연속 신호에서 ICA를 다시 학습하는 실험은 별개입니다.


In [ ]:
# 1. 새 계정의 Drive 연결 및 GPU 확인
import os, torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요: 런타임 → 런타임 유형 변경 → GPU'
DRIVE_ROOT = Path('/content/drive/MyDrive/BCI_Research')
assert DRIVE_ROOT.is_dir(), f'새 계정 Drive에 폴더가 없습니다: {DRIVE_ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Drive:', DRIVE_ROOT)


In [ ]:
# 2. 검증된 코드와 제외 목록을 Colab 런타임에 설치
import base64, hashlib
CODE_DIR = Path("/content/mi_bci_rigorous")
CODE_DIR.mkdir(exist_ok=True)
(CODE_DIR / 'ablation_study_v5.py').write_bytes(base64.b64decode('...'))
assert hashlib.sha256((CODE_DIR / 'ablation_study_v5.py').read_bytes()).hexdigest() == 'edb9ad5dca471de6e4d90738699e6d9f67428f2ca511d93988761d22d02d5761'
(CODE_DIR / 'rigorous_loso.py').write_bytes(base64.b64decode('...'))
assert hashlib.sha256((CODE_DIR / 'rigorous_loso.py').read_bytes()).hexdigest() == 'cad5e0de25703806a9225635806676add5b9376aad4fc2e5f621138ae92cb40b'
(CODE_DIR / 'rigorous_group5.py').write_bytes(base64.b64decode('IiIiRml2ZS1mb2xkLCBzdWJqZWN0LWRpc2pvaW50IHBhaXJlZCBjb250YW1pbmF0aW9uIGF1ZGl0LgoKVGhlIHNhbWUgZml4ZWQgb3V0ZXIgZm9sZHMgYW5kIGNsZWFuIHRlc3QgdHJpYWxzIGFyZSB1c2VkIGZvciBhbGwgbW9kZWxzLAp0cmFpbmluZyBjb25kaXRpb25zLCBhbmQgc2VlZHMuIE9uZSBjbGVhbiB2YWxpZGF0aW9uIHN1YmplY3QgaXMgc2VsZWN0ZWQgZnJvbQp0aGUgdHJhaW5pbmcgc2lkZSBvZiBlYWNoIGZvbGQuIEVhY2ggdGVzdCBzdWJqZWN0IHJlY2VpdmVzIG9uZSBwcmVkaWN0aW9uIHBlcgpzZWVkIGFuZCBjb25kaXRpb24sIGFmdGVyIG1vZGVsIHNlbGVjdGlvbiBvbiB2YWxpZGF0aW9uIGRhdGEuCgpUaGlzIGlzIGEgc2VwYXJhdGUgcHJvdG9jb2wgZnJvbSByaWdvcm91c19sb3NvLnB5LiBJdHMgcmVzdWx0cyBtdXN0IG5vdCBiZQpwb29sZWQgd2l0aCBMT1NPIHJlc3VsdHMuIEhERjUgZXBvY2hzIGFyZSBzdHJlYW1lZCB0byBib3VuZCBob3N0IFJBTSB1c2UuCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBjc3YKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IHN5cwppbXBvcnQgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0IGYxX3Njb3JlCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgQ29uY2F0RGF0YXNldCwgRGF0YUxvYWRlcgoKZnJvbSBhYmxhdGlvbl9zdHVkeV92NSBpbXBvcnQgKENGRywgTU9ERUxfQ0xBU1NFUywgX21ldHJpY3MsIGV2YWx1YXRlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcGlja19kZXZpY2UsIHNldF9zZWVkLCB0cmFpbl9lcG9jaCkKZnJvbSByaWdvcm91c19sb3NvIGltcG9ydCAoREVGQVVMVF9EQVRBLCBERUZBVUxUX0VYQ0xVU0lPTlMsIEZJRUxEUywKICAgICAgICAgICAgICAgICAgICAgICAgICAgbG9hZF9tYW5pZmVzdCwgbWFrZV9kYXRhc2V0KQoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CkRFRkFVTFRfT1VUUFVUID0gUk9PVCAvICJCQ0lfUmVzZWFyY2gvcmVzdWx0cy9yaWdvcm91c19ncm91cDUiCkZPTERfU0VFRCA9IDIwMjYxMDExCkdST1VQX0ZJRUxEUyA9IFsib3V0ZXJfZm9sZCIsICpGSUVMRFMsICJmb2xkX2VsYXBzZWRfc2Vjb25kcyJdCgoKZGVmIG91dGVyX2ZvbGRzKGVsaWdpYmxlKToKICAgICIiIkZpeGVkLCByZXByb2R1Y2libGUgZml2ZS13YXkgcGFydGl0aW9uIG9mIHRoZSA0OSB0ZXN0IHN1YmplY3RzLiIiIgogICAgc2h1ZmZsZWQgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoRk9MRF9TRUVEKS5wZXJtdXRhdGlvbihzb3J0ZWQoZWxpZ2libGUpKQogICAgcmV0dXJuIFtzb3J0ZWQoaW50KHMpIGZvciBzIGluIHBhcnQpIGZvciBwYXJ0IGluIG5wLmFycmF5X3NwbGl0KHNodWZmbGVkLCA1KV0KCgpkZWYgc3BsaXRfZ3JvdXAoZm9sZF9pZHgsIGZvbGRzLCBlbGlnaWJsZSwgcG9vbCwgc2VlZCk6CiAgICB0ZXN0X3NpZHMgPSBmb2xkc1tmb2xkX2lkeF0KICAgIGNhbmRpZGF0ZXMgPSBzb3J0ZWQoc2V0KGVsaWdpYmxlKSAtIHNldCh0ZXN0X3NpZHMpKQogICAgdmFsaWRhdGlvbl9zaWQgPSBjYW5kaWRhdGVzWyhzZWVkICsgZm9sZF9pZHgpICUgbGVuKGNhbmRpZGF0ZXMpXQogICAgdHJhaW5fc2lkcyA9IHNvcnRlZChzZXQocG9vbCkgLSBzZXQodGVzdF9zaWRzKSAtIHt2YWxpZGF0aW9uX3NpZH0pCiAgICBhc3NlcnQgbm90IChzZXQodHJhaW5fc2lkcykgJiBzZXQodGVzdF9zaWRzKSkKICAgIGFzc2VydCB2YWxpZGF0aW9uX3NpZCBub3QgaW4gdGVzdF9zaWRzIGFuZCB2YWxpZGF0aW9uX3NpZCBub3QgaW4gdHJhaW5fc2lkcwogICAgcmV0dXJuIHRyYWluX3NpZHMsIHZhbGlkYXRpb25fc2lkLCB0ZXN0X3NpZHMKCgpkZWYgcnVuX2dyb3VwKGZvbGRfaWR4LCBmb2xkcywgZWxpZ2libGUsIHBvb2wsIGV4Y2x1c2lvbnMsIGRhdGFfZGlyLAogICAgICAgICAgICAgIG1vZGVsX3R5cGUsIGNvbmRpdGlvbiwgc2VlZCwgbWF4X2Vwb2NocywgcGF0aWVuY2UsIGRldmljZSk6CiAgICB0cmFpbl9zaWRzLCB2YWxpZGF0aW9uX3NpZCwgdGVzdF9zaWRzID0gc3BsaXRfZ3JvdXAoCiAgICAgICAgZm9sZF9pZHgsIGZvbGRzLCBlbGlnaWJsZSwgcG9vbCwgc2VlZCkKICAgIHNldF9zZWVkKHNlZWQgKiAxMDAwMDAgKyBmb2xkX2lkeCkKICAgIHRyYWluaW5nID0gQ29uY2F0RGF0YXNldChbCiAgICAgICAgbWFrZV9kYXRhc2V0KHMsIGRhdGFfZGlyLCBleGNsdXNpb25zLCBjb25kaXRpb24gPT0gImNsZWFuX3RyYWluIiwgbW9kZWxfdHlwZSkKICAgICAgICBmb3IgcyBpbiB0cmFpbl9zaWRzCiAgICBdKQogICAgdmFsaWRhdGlvbiA9IG1ha2VfZGF0YXNldCh2YWxpZGF0aW9uX3NpZCwgZGF0YV9kaXIsIGV4Y2x1c2lvbnMsIFRydWUsIG1vZGVsX3R5cGUpCiAgICB0cmFpbl9sb2FkZXIgPSBEYXRhTG9hZGVyKHRyYWluaW5nLCBiYXRjaF9zaXplPTMyLCBzaHVmZmxlPVRydWUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG51bV93b3JrZXJzPTAsIHBpbl9tZW1vcnk9ZGV2aWNlLnR5cGUgPT0gImN1ZGEiKQogICAgdmFsX2xvYWRlciA9IERhdGFMb2FkZXIodmFsaWRhdGlvbiwgYmF0Y2hfc2l6ZT02NCwgc2h1ZmZsZT1GYWxzZSkKCiAgICBtb2RlbCA9IE1PREVMX0NMQVNTRVNbbW9kZWxfdHlwZV0oZGljdChDRkcpKS50byhkZXZpY2UpCiAgICBvcHRpbWl6ZXIgPSB0b3JjaC5vcHRpbS5BZGFtKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9MWUtMywgd2VpZ2h0X2RlY2F5PTFlLTQpCiAgICBzY2FsZXIgPSB0b3JjaC5hbXAuR3JhZFNjYWxlcigiY3VkYSIpIGlmIGRldmljZS50eXBlID09ICJjdWRhIiBlbHNlIE5vbmUKICAgIGJlc3RfZjEsIGJlc3Rfc3RhdGUsIGJlc3RfZXBvY2gsIHN0YWxlID0gLTEuMCwgTm9uZSwgMCwgMAogICAgZXBvY2hzX3J1biA9IDAKICAgIGZvciBlcG9jaCBpbiByYW5nZSgxLCBtYXhfZXBvY2hzICsgMSk6CiAgICAgICAgdHJhaW5fZXBvY2gobW9kZWwsIHRyYWluX2xvYWRlciwgb3B0aW1pemVyLCBzY2FsZXIsIGRldmljZSkKICAgICAgICBwcmVkLCB0cnVlID0gZXZhbHVhdGUobW9kZWwsIHZhbF9sb2FkZXIsIGRldmljZSkKICAgICAgICBzY29yZSA9IGYxX3Njb3JlKHRydWUsIHByZWQsIGF2ZXJhZ2U9Im1hY3JvIiwgemVyb19kaXZpc2lvbj0wKQogICAgICAgIGVwb2Noc19ydW4gPSBlcG9jaAogICAgICAgIGlmIHNjb3JlID4gYmVzdF9mMToKICAgICAgICAgICAgYmVzdF9mMSA9IHNjb3JlCiAgICAgICAgICAgIGJlc3Rfc3RhdGUgPSB7a2V5OiB2YWx1ZS5kZXRhY2goKS5jcHUoKS5jbG9uZSgpCiAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGtleSwgdmFsdWUgaW4gbW9kZWwuc3RhdGVfZGljdCgpLml0ZW1zKCl9CiAgICAgICAgICAgIGJlc3RfZXBvY2gsIHN0YWxlID0gZXBvY2gsIDAKICAgICAgICBlbHNlOgogICAgICAgICAgICBzdGFsZSArPSAxCiAgICAgICAgaWYgZXBvY2ggPT0gMSBvciBlcG9jaCAlIDUgPT0gMCBvciBzdGFsZSA+PSBwYXRpZW5jZToKICAgICAgICAgICAgcHJpbnQoZiIgICAgZm9sZCB7Zm9sZF9pZHggKyAxfS81IGVwb2NoIHtlcG9jaH0ve21heF9lcG9jaHN9ICIKICAgICAgICAgICAgICAgICAgZiJ2YWxpZGF0aW9uX0YxPXtzY29yZTouNGZ9IGJlc3RfZXBvY2g9e2Jlc3RfZXBvY2h9ICIKICAgICAgICAgICAgICAgICAgZiJwYXRpZW5jZT17c3RhbGV9L3twYXRpZW5jZX0iLCBmbHVzaD1UcnVlKQogICAgICAgIGlmIHN0YWxlID49IHBhdGllbmNlOgogICAgICAgICAgICBicmVhawoKICAgIG1vZGVsLmxvYWRfc3RhdGVfZGljdChiZXN0X3N0YXRlKQogICAgcm93cyA9IFtdCiAgICBmb3Igc2lkIGluIHRlc3Rfc2lkczoKICAgICAgICB0ZXN0aW5nID0gbWFrZV9kYXRhc2V0KHNpZCwgZGF0YV9kaXIsIGV4Y2x1c2lvbnMsIFRydWUsIG1vZGVsX3R5cGUpCiAgICAgICAgdGVzdF9sb2FkZXIgPSBEYXRhTG9hZGVyKHRlc3RpbmcsIGJhdGNoX3NpemU9NjQsIHNodWZmbGU9RmFsc2UpCiAgICAgICAgcHJlZCwgdHJ1ZSA9IGV2YWx1YXRlKG1vZGVsLCB0ZXN0X2xvYWRlciwgZGV2aWNlKQogICAgICAgIHJvdyA9IF9tZXRyaWNzKHNpZCwgcHJlZCwgdHJ1ZSkKICAgICAgICByb3cucG9wKCJjb25mdXNpb24iLCBOb25lKQogICAgICAgIHJvdy51cGRhdGUob3V0ZXJfZm9sZD1mb2xkX2lkeCArIDEsIHZhbGlkYXRpb25fc2lkPXZhbGlkYXRpb25fc2lkLAogICAgICAgICAgICAgICAgICAgc2VlZD1zZWVkLCBtb2RlbF90eXBlPW1vZGVsX3R5cGUsIGNvbmRpdGlvbj1jb25kaXRpb24sCiAgICAgICAgICAgICAgICAgICBuX3RyYWluX3RyaWFscz1sZW4odHJhaW5pbmcpLCBuX3ZhbF90cmlhbHM9bGVuKHZhbGlkYXRpb24pLAogICAgICAgICAgICAgICAgICAgYmVzdF9lcG9jaD1iZXN0X2Vwb2NoLCBlcG9jaHNfcnVuPWVwb2Noc19ydW4pCiAgICAgICAgcm93cy5hcHBlbmQocm93KQogICAgcmV0dXJuIHJvd3MKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29uZGl0aW9uIiwgY2hvaWNlcz1bImRpcnR5X3RyYWluIiwgImNsZWFuX3RyYWluIl0sIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsX3R5cGUiLCBjaG9pY2VzPXNvcnRlZChNT0RFTF9DTEFTU0VTKSwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZCIsIHR5cGU9aW50LCBkZWZhdWx0PTQyKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mb2xkcyIsIHR5cGU9aW50LCBuYXJncz0iKyIsIGRlZmF1bHQ9bGlzdChyYW5nZSgxLCA2KSkpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRhdGFfZGlyIiwgdHlwZT1QYXRoLCBkZWZhdWx0PURFRkFVTFRfREFUQSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXhjbHVkZV9qc29uIiwgdHlwZT1QYXRoLCBkZWZhdWx0PURFRkFVTFRfRVhDTFVTSU9OUykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tb3V0X2RpciIsIHR5cGU9UGF0aCwgZGVmYXVsdD1ERUZBVUxUX09VVFBVVCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbWF4X2Vwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTIwMCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tcGF0aWVuY2UiLCB0eXBlPWludCwgZGVmYXVsdD0yMCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgY2hvaWNlcz1bImNwdSIsICJjdWRhIiwgIm1wcyJdKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kcnlfcnVuIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICBpZiBhcmdzLm1heF9lcG9jaHMgPCAxIG9yIGFyZ3MucGF0aWVuY2UgPCAxOgogICAgICAgIHBhcnNlci5lcnJvcigibWF4X2Vwb2NocyBhbmQgcGF0aWVuY2UgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICBpZiBsZW4oc2V0KGFyZ3MuZm9sZHMpKSAhPSBsZW4oYXJncy5mb2xkcykgb3Igbm90IHNldChhcmdzLmZvbGRzKSA8PSBzZXQocmFuZ2UoMSwgNikpOgogICAgICAgIHBhcnNlci5lcnJvcigiLS1mb2xkcyBtdXN0IGNvbnRhaW4gZGlzdGluY3QgaW50ZWdlcnMgZnJvbSAxIHRvIDUiKQoKICAgIGV4Y2x1c2lvbnMsIHBvb2wsIGVsaWdpYmxlID0gbG9hZF9tYW5pZmVzdChhcmdzLmV4Y2x1ZGVfanNvbiwgYXJncy5kYXRhX2RpcikKICAgIGZvbGRzID0gb3V0ZXJfZm9sZHMoZWxpZ2libGUpCiAgICBpZiBhcmdzLmRyeV9ydW46CiAgICAgICAgZm9yIGZvbGQgaW4gYXJncy5mb2xkczoKICAgICAgICAgICAgdHJhaW4sIHZhbCwgdGVzdCA9IHNwbGl0X2dyb3VwKGZvbGQgLSAxLCBmb2xkcywgZWxpZ2libGUsIHBvb2wsIGFyZ3Muc2VlZCkKICAgICAgICAgICAgcHJpbnQoZiJmb2xkPXtmb2xkfS81IHRlc3Q9e3Rlc3R9IHZhbGlkYXRpb249c3t2YWw6MDJkfSAiCiAgICAgICAgICAgICAgICAgIGYidHJhaW5fc3ViamVjdHM9e2xlbih0cmFpbil9IikKICAgICAgICByZXR1cm4gMAoKICAgIGRldmljZSA9IHBpY2tfZGV2aWNlKGFyZ3MubW9kZWxfdHlwZSwgYXJncy5kZXZpY2UpCiAgICBpZiBkZXZpY2UudHlwZSA9PSAibXBzIiBhbmQgYXJncy5tb2RlbF90eXBlICE9ICJlbWdfb25seSI6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJNUFMgaXMgZGlzYWxsb3dlZCBmb3IgRUVHTmV0IGNvbmRpdGlvbnMiKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiIGFuZCBub3QgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoIkNVREEgcmVxdWVzdGVkIGJ1dCB1bmF2YWlsYWJsZSIpCgogICAgYXJncy5vdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIG91dHB1dCA9IGFyZ3Mub3V0X2RpciAvIGYie2FyZ3MuY29uZGl0aW9ufV97YXJncy5tb2RlbF90eXBlfV9zZWVke2FyZ3Muc2VlZH0uY3N2IgogICAgY29uZmlnX3BhdGggPSBvdXRwdXQud2l0aF9zdWZmaXgoIi5jb25maWcuanNvbiIpCiAgICBjb25maWcgPSB7CiAgICAgICAgInByb3RvY29sIjogImZpeGVkX3N1YmplY3RfZ3JvdXA1IiwgImZvbGRfc2VlZCI6IEZPTERfU0VFRCwKICAgICAgICAiZm9sZHMiOiBmb2xkcywgImNvbmRpdGlvbiI6IGFyZ3MuY29uZGl0aW9uLCAibW9kZWxfdHlwZSI6IGFyZ3MubW9kZWxfdHlwZSwKICAgICAgICAic2VlZCI6IGFyZ3Muc2VlZCwgIm1heF9lcG9jaHMiOiBhcmdzLm1heF9lcG9jaHMsICJwYXRpZW5jZSI6IGFyZ3MucGF0aWVuY2UsCiAgICAgICAgInBvb2wiOiBwb29sLCAiZWxpZ2libGUiOiBlbGlnaWJsZSwgImNsZWFuX2VlZ19yZW5vcm1hbGl6YXRpb24iOiBUcnVlLAogICAgICAgICJleGNsdXNpb25zX3NoYTI1NiI6IGhhc2hsaWIuc2hhMjU2KGFyZ3MuZXhjbHVkZV9qc29uLnJlYWRfYnl0ZXMoKSkuaGV4ZGlnZXN0KCksCiAgICAgICAgInJ1bm5lcl9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihQYXRoKF9fZmlsZV9fKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICJsb2FkZXJfc2hhMjU2IjogaGFzaGxpYi5zaGEyNTYoUGF0aChzeXMubW9kdWxlc1sicmlnb3JvdXNfbG9zbyJdLl9fZmlsZV9fKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICJtb2RlbF9zaGEyNTYiOiBoYXNobGliLnNoYTI1NihQYXRoKHN5cy5tb2R1bGVzWyJhYmxhdGlvbl9zdHVkeV92NSJdLl9fZmlsZV9fKS5yZWFkX2J5dGVzKCkpLmhleGRpZ2VzdCgpLAogICAgICAgICJkYXRhX2RpciI6IHN0cihhcmdzLmRhdGFfZGlyLnJlc29sdmUoKSksCiAgICB9CiAgICBpZiBjb25maWdfcGF0aC5leGlzdHMoKToKICAgICAgICBpZiBqc29uLmxvYWRzKGNvbmZpZ19wYXRoLnJlYWRfdGV4dCgpKSAhPSBjb25maWc6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlZnVzaW5nIHRvIHJlc3VtZSB3aXRoIGRpZmZlcmVudCBzZXR0aW5nczoge291dHB1dH0iKQogICAgZWxzZToKICAgICAgICBpZiBvdXRwdXQuZXhpc3RzKCk6CiAgICAgICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIlJlc3VsdHMgZXhpc3Qgd2l0aG91dCBhIGNvbmZpZyBtYW5pZmVzdDoge291dHB1dH0iKQogICAgICAgIGNvbmZpZ19wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhjb25maWcsIGluZGVudD0yKSkKCiAgICBjb21wbGV0ZWQgPSBzZXQoKQogICAgaWYgb3V0cHV0LmV4aXN0cygpOgogICAgICAgIHdpdGggb3V0cHV0Lm9wZW4obmV3bGluZT0iIikgYXMgc3RyZWFtOgogICAgICAgICAgICByb3dzID0gbGlzdChjc3YuRGljdFJlYWRlcihzdHJlYW0pKQogICAgICAgIGZvciBmb2xkIGluIHJhbmdlKDEsIDYpOgogICAgICAgICAgICBhY3R1YWwgPSB7aW50KHJvd1sic2lkIl0pIGZvciByb3cgaW4gcm93cyBpZiBpbnQocm93WyJvdXRlcl9mb2xkIl0pID09IGZvbGR9CiAgICAgICAgICAgIGV4cGVjdGVkID0gc2V0KGZvbGRzW2ZvbGQgLSAxXSkKICAgICAgICAgICAgaWYgYWN0dWFsIGFuZCBhY3R1YWwgIT0gZXhwZWN0ZWQ6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJQYXJ0aWFsIG9yIGluY29uc2lzdGVudCBmb2xkIHtmb2xkfToge291dHB1dH0iKQogICAgICAgICAgICBpZiBhY3R1YWw6CiAgICAgICAgICAgICAgICBjb21wbGV0ZWQuYWRkKGZvbGQpCiAgICAgICAgaWYgbGVuKHJvd3MpICE9IHN1bShsZW4oZm9sZHNbZm9sZCAtIDFdKSBmb3IgZm9sZCBpbiBjb21wbGV0ZWQpOgogICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoZiJEdXBsaWNhdGUgb3IgdW5leHBlY3RlZCByZXN1bHQgcm93czoge291dHB1dH0iKQoKICAgIGZvbGRfdGltZXMgPSB7fQogICAgaWYgb3V0cHV0LmV4aXN0cygpOgogICAgICAgIGZvciByb3cgaW4gcm93czoKICAgICAgICAgICAgaWYgcm93LmdldCgiZm9sZF9lbGFwc2VkX3NlY29uZHMiKToKICAgICAgICAgICAgICAgIGZvbGRfdGltZXNbaW50KHJvd1sib3V0ZXJfZm9sZCJdKV0gPSBmbG9hdChyb3dbImZvbGRfZWxhcHNlZF9zZWNvbmRzIl0pCgogICAgcHJpbnQoZiJ7YXJncy5jb25kaXRpb259IC8ge2FyZ3MubW9kZWxfdHlwZX0gLyBzZWVkIHthcmdzLnNlZWR9OiAiCiAgICAgICAgICBmIuyZhOujjCB7bGVuKGNvbXBsZXRlZCl9LzUgZm9sZHMsIOuCqOydjCB7NSAtIGxlbihjb21wbGV0ZWQpfSIsIGZsdXNoPVRydWUpCiAgICBmb3IgZm9sZCBpbiBhcmdzLmZvbGRzOgogICAgICAgIGlmIGZvbGQgaW4gY29tcGxldGVkOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIF8sIHZhbGlkYXRpb25fc2lkLCB0ZXN0X3NpZHMgPSBzcGxpdF9ncm91cCgKICAgICAgICAgICAgZm9sZCAtIDEsIGZvbGRzLCBlbGlnaWJsZSwgcG9vbCwgYXJncy5zZWVkKQogICAgICAgIHByaW50KGYiW3tsZW4oY29tcGxldGVkKX0vNV0gZm9sZCB7Zm9sZH0g7Iuc7J6RICIKICAgICAgICAgICAgICBmIih0ZXN0PXt0ZXN0X3NpZHN9LCB2YWxpZGF0aW9uPXN7dmFsaWRhdGlvbl9zaWQ6MDJkfSkiLCBmbHVzaD1UcnVlKQogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgcm93cyA9IHJ1bl9ncm91cChmb2xkIC0gMSwgZm9sZHMsIGVsaWdpYmxlLCBwb29sLCBleGNsdXNpb25zLCBhcmdzLmRhdGFfZGlyLAogICAgICAgICAgICAgICAgICAgICAgICAgYXJncy5tb2RlbF90eXBlLCBhcmdzLmNvbmRpdGlvbiwgYXJncy5zZWVkLAogICAgICAgICAgICAgICAgICAgICAgICAgYXJncy5tYXhfZXBvY2hzLCBhcmdzLnBhdGllbmNlLCBkZXZpY2UpCiAgICAgICAgZWxhcHNlZCA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSBzdGFydGVkCiAgICAgICAgZm9yIHJvdyBpbiByb3dzOgogICAgICAgICAgICByb3dbImZvbGRfZWxhcHNlZF9zZWNvbmRzIl0gPSByb3VuZChlbGFwc2VkLCAxKQogICAgICAgIG5ld19maWxlID0gbm90IG91dHB1dC5leGlzdHMoKQogICAgICAgIHdpdGggb3V0cHV0Lm9wZW4oImEiLCBuZXdsaW5lPSIiKSBhcyBzdHJlYW06CiAgICAgICAgICAgIHdyaXRlciA9IGNzdi5EaWN0V3JpdGVyKHN0cmVhbSwgZmllbGRuYW1lcz1HUk9VUF9GSUVMRFMpCiAgICAgICAgICAgIGlmIG5ld19maWxlOgogICAgICAgICAgICAgICAgd3JpdGVyLndyaXRlaGVhZGVyKCkKICAgICAgICAgICAgd3JpdGVyLndyaXRlcm93cyh7a2V5OiByb3dba2V5XSBmb3Iga2V5IGluIEdST1VQX0ZJRUxEU30gZm9yIHJvdyBpbiByb3dzKQogICAgICAgIGNvbXBsZXRlZC5hZGQoZm9sZCkKICAgICAgICBmb2xkX3RpbWVzW2ZvbGRdID0gZWxhcHNlZAogICAgICAgIHJlbWFpbmluZyA9IGxlbihzZXQocmFuZ2UoMSwgNikpIC0gY29tcGxldGVkKQogICAgICAgIGVzdGltYXRlZF9taW51dGVzID0gcmVtYWluaW5nICogbnAubWVhbihsaXN0KGZvbGRfdGltZXMudmFsdWVzKCkpKSAvIDYwCiAgICAgICAgcHJpbnQoZiJbe2xlbihjb21wbGV0ZWQpfS81XSBmb2xkIHtmb2xkfSDsmYTro4wgIgogICAgICAgICAgICAgIGYic3ViamVjdHM9e2xlbihyb3dzKX0sIGJlc3RfZXBvY2g9e3Jvd3NbMF1bJ2Jlc3RfZXBvY2gnXX0sICIKICAgICAgICAgICAgICBmImVsYXBzZWQ9e2VsYXBzZWQ6LjBmfXM7IOuCqOydjCB7cmVtYWluaW5nfTsgIgogICAgICAgICAgICAgIGYi7J20IOyhsO2VqSDsmIjsg4Eg7J6U7JesIHtlc3RpbWF0ZWRfbWludXRlczouMGZ967aEIiwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHN5cy5leGl0KG1haW4oKSkK'))
assert hashlib.sha256((CODE_DIR / 'rigorous_group5.py').read_bytes()).hexdigest() == 'ac65c689419a3ed2883ae10e28ee214a43e5b74c0cc32d7d1aba99a0a58e3f4f'
(CODE_DIR / 'analyze_rigorous_loso.py').write_bytes(base64.b64decode('IiIiQW5hbHl6ZSBjb21wbGV0ZSBwYWlyZWQgcmlnb3JvdXNfbG9zbyBydW5zIHdpdGhvdXQgdHJlYXRpbmcgc2VlZHMgYXMgc3ViamVjdHMuIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNjaXB5LnN0YXRzIGltcG9ydCB3aWxjb3hvbgoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CkRFRkFVTFRfUkVTVUxUUyA9IFJPT1QgLyAiQkNJX1Jlc2VhcmNoL3Jlc3VsdHMvcmlnb3JvdXNfbG9zbyIKQ09ORElUSU9OUyA9ICgiZGlydHlfdHJhaW4iLCAiY2xlYW5fdHJhaW4iKQpNT0RFTFMgPSAoImVlZ19vbmx5IiwgImVtZ19vbmx5IiwgImZ1c2lvbiIpCk1FVFJJQ1MgPSAoImFjY3VyYWN5IiwgImthcHBhIikKCgpkZWYgbG9hZF9jb21wbGV0ZShvdXRfZGlyLCBzZWVkcywgZXhwZWN0ZWRfc3ViamVjdHM9NDkpOgogICAgZnJhbWVzID0gW10KICAgIGZvciBjb25kaXRpb24gaW4gQ09ORElUSU9OUzoKICAgICAgICBmb3IgbW9kZWwgaW4gTU9ERUxTOgogICAgICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgICAgIHBhdGggPSBvdXRfZGlyIC8gZiJ7Y29uZGl0aW9ufV97bW9kZWx9X3NlZWR7c2VlZH0uY3N2IgogICAgICAgICAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUmVxdWlyZWQgcnVuIGlzIG1pc3Npbmc6IHtwYXRofSIpCiAgICAgICAgICAgICAgICBkZiA9IHBkLnJlYWRfY3N2KHBhdGgpCiAgICAgICAgICAgICAgICBpZiAobGVuKGRmKSAhPSBleHBlY3RlZF9zdWJqZWN0cyBvciBkZi5zaWQubnVuaXF1ZSgpICE9IGV4cGVjdGVkX3N1YmplY3RzCiAgICAgICAgICAgICAgICAgICAgICAgIG9yIHNldChkZi5jb25kaXRpb24pICE9IHtjb25kaXRpb259CiAgICAgICAgICAgICAgICAgICAgICAgIG9yIHNldChkZi5tb2RlbF90eXBlKSAhPSB7bW9kZWx9CiAgICAgICAgICAgICAgICAgICAgICAgIG9yIHNldChkZi5zZWVkKSAhPSB7c2VlZH0pOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbmNvbXBsZXRlIG9yIGluY29uc2lzdGVudCBydW46IHtwYXRofSIpCiAgICAgICAgICAgICAgICBmcmFtZXMuYXBwZW5kKGRmKQogICAgYWxsX3Jvd3MgPSBwZC5jb25jYXQoZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgIGlkcyA9IHt0dXBsZShzb3J0ZWQoZi5zaWQpKSBmb3IgZiBpbiBmcmFtZXN9CiAgICBpZiBsZW4oaWRzKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNvbmRpdGlvbnMgZG8gbm90IGNvbnRhaW4gdGhlIHNhbWUgdGVzdCBzdWJqZWN0cyIpCiAgICBncm91cGVkID0gYWxsX3Jvd3MuZ3JvdXBieShbInNlZWQiLCAic2lkIl0pCiAgICBpZiBncm91cGVkLnZhbGlkYXRpb25fc2lkLm51bmlxdWUoKS5tYXgoKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlBhaXJlZCBydW5zIHNlbGVjdGVkIGRpZmZlcmVudCB2YWxpZGF0aW9uIHN1YmplY3RzIikKICAgIGlmIGdyb3VwZWQubl90cmlhbHMubnVuaXF1ZSgpLm1heCgpICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUGFpcmVkIHJ1bnMgdXNlZCBkaWZmZXJlbnQgY2xlYW4gdGVzdC10cmlhbCBjb3VudHMiKQogICAgcmV0dXJuIGFsbF9yb3dzCgoKZGVmIHBhaXJlZF9yZXN1bHQoZGYsIG51bWVyYXRvciwgZGVub21pbmF0b3IsIG1ldHJpYywgYm9vdHN0cmFwPTEwMDAwKToKICAgICIiIkF2ZXJhZ2UgZWFjaCBzdWJqZWN0IGFjcm9zcyBzZWVkcywgdGhlbiB0ZXN0IHBhaXJlZCBzdWJqZWN0IGRpZmZlcmVuY2VzLiIiIgogICAgcGVyX3N1YmplY3QgPSBkZi5ncm91cGJ5KFsiY29uZGl0aW9uIiwgIm1vZGVsX3R5cGUiLCAic2lkIl0pW21ldHJpY10ubWVhbigpCiAgICBhID0gcGVyX3N1YmplY3QueHMobnVtZXJhdG9yWzoyXSkuc29ydF9pbmRleCgpCiAgICBiID0gcGVyX3N1YmplY3QueHMoZGVub21pbmF0b3JbOjJdKS5zb3J0X2luZGV4KCkKICAgIGlmIG5vdCBhLmluZGV4LmVxdWFscyhiLmluZGV4KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTdWJqZWN0IHBhaXJpbmcgZmFpbGVkIikKICAgIGRlbHRhID0gKGEgLSBiKS50b19udW1weShkdHlwZT1mbG9hdCkKICAgIHAgPSBmbG9hdCh3aWxjb3hvbihkZWx0YSkucHZhbHVlKSBpZiBucC5hbnkoZGVsdGEgIT0gMCkgZWxzZSAxLjAKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyg0MikKICAgIGRyYXdzID0gcm5nLmNob2ljZShkZWx0YSwgc2l6ZT0oYm9vdHN0cmFwLCBsZW4oZGVsdGEpKSwgcmVwbGFjZT1UcnVlKS5tZWFuKGF4aXM9MSkKICAgIGxvdywgaGlnaCA9IG5wLnF1YW50aWxlKGRyYXdzLCBbMC4wMjUsIDAuOTc1XSkKICAgIHJldHVybiB7Im5fc3ViamVjdHMiOiBsZW4oZGVsdGEpLCAibWVhbl9hIjogZmxvYXQoYS5tZWFuKCkpLAogICAgICAgICAgICAibWVhbl9iIjogZmxvYXQoYi5tZWFuKCkpLCAibWVhbl9wYWlyZWRfZGVsdGEiOiBmbG9hdChkZWx0YS5tZWFuKCkpLAogICAgICAgICAgICAiYm9vdHN0cmFwXzk1Y2kiOiBbZmxvYXQobG93KSwgZmxvYXQoaGlnaCldLCAid2lsY294b25fcCI6IHAsCiAgICAgICAgICAgICJwb3NpdGl2ZV9zdWJqZWN0cyI6IGludCgoZGVsdGEgPiAwKS5zdW0oKSksCiAgICAgICAgICAgICJuZWdhdGl2ZV9zdWJqZWN0cyI6IGludCgoZGVsdGEgPCAwKS5zdW0oKSksCiAgICAgICAgICAgICJ6ZXJvX3N1YmplY3RzIjogaW50KChkZWx0YSA9PSAwKS5zdW0oKSl9CgoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dF9kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9REVGQVVMVF9SRVNVTFRTKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkcyIsIHR5cGU9aW50LCBuYXJncz0iKyIsIGRlZmF1bHQ9WzQyLCAxMzM3LCAyMDI0XSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYm9vdHN0cmFwIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAwMDApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgaWYgbGVuKHNldChhcmdzLnNlZWRzKSkgIT0gbGVuKGFyZ3Muc2VlZHMpIG9yIGFyZ3MuYm9vdHN0cmFwIDwgMTAwOgogICAgICAgIHBhcnNlci5lcnJvcigiU2VlZHMgbXVzdCBiZSB1bmlxdWUgYW5kIGJvb3RzdHJhcCBtdXN0IGJlIGF0IGxlYXN0IDEwMCIpCiAgICBkZiA9IGxvYWRfY29tcGxldGUoYXJncy5vdXRfZGlyLCBhcmdzLnNlZWRzKQoKICAgIGFuYWx5c2VzID0ge30KICAgIGZvciBtZXRyaWMgaW4gTUVUUklDUzoKICAgICAgICBmb3IgbW9kZWwgaW4gTU9ERUxTOgogICAgICAgICAgICBrZXkgPSBmInRyYWluaW5nX2NsZWFudXBfe21vZGVsfV97bWV0cmljfSIKICAgICAgICAgICAgYW5hbHlzZXNba2V5XSA9IHBhaXJlZF9yZXN1bHQoCiAgICAgICAgICAgICAgICBkZiwgKCJjbGVhbl90cmFpbiIsIG1vZGVsKSwgKCJkaXJ0eV90cmFpbiIsIG1vZGVsKSwKICAgICAgICAgICAgICAgIG1ldHJpYywgYXJncy5ib290c3RyYXApCiAgICAgICAgZm9yIGNvbmRpdGlvbiBpbiBDT05ESVRJT05TOgogICAgICAgICAgICBrZXkgPSBmImZ1c2lvbl9nYWluX3tjb25kaXRpb259X3ttZXRyaWN9IgogICAgICAgICAgICBhbmFseXNlc1trZXldID0gcGFpcmVkX3Jlc3VsdCgKICAgICAgICAgICAgICAgIGRmLCAoY29uZGl0aW9uLCAiZnVzaW9uIiksIChjb25kaXRpb24sICJlZWdfb25seSIpLAogICAgICAgICAgICAgICAgbWV0cmljLCBhcmdzLmJvb3RzdHJhcCkKCiAgICAjIERpZmZlcmVuY2UgaW4gZnVzaW9uIGFkdmFudGFnZSBhdHRyaWJ1dGFibGUgdG8gdHJhaW5pbmctdHJpYWwgY2xlYW51cC4KICAgIGZvciBtZXRyaWMgaW4gTUVUUklDUzoKICAgICAgICBwaXZvdCA9IGRmLmdyb3VwYnkoWyJzaWQiLCAiY29uZGl0aW9uIiwgIm1vZGVsX3R5cGUiXSlbbWV0cmljXS5tZWFuKCkudW5zdGFjaygKICAgICAgICAgICAgWyJjb25kaXRpb24iLCAibW9kZWxfdHlwZSJdKQogICAgICAgIGRpcnR5ID0gcGl2b3RbKCJkaXJ0eV90cmFpbiIsICJmdXNpb24iKV0gLSBwaXZvdFsoImRpcnR5X3RyYWluIiwgImVlZ19vbmx5IildCiAgICAgICAgY2xlYW4gPSBwaXZvdFsoImNsZWFuX3RyYWluIiwgImZ1c2lvbiIpXSAtIHBpdm90WygiY2xlYW5fdHJhaW4iLCAiZWVnX29ubHkiKV0KICAgICAgICBkZWx0YSA9IChjbGVhbiAtIGRpcnR5KS50b19udW1weShkdHlwZT1mbG9hdCkKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNDIpCiAgICAgICAgZHJhd3MgPSBybmcuY2hvaWNlKGRlbHRhLCBzaXplPShhcmdzLmJvb3RzdHJhcCwgbGVuKGRlbHRhKSksIHJlcGxhY2U9VHJ1ZSkubWVhbihheGlzPTEpCiAgICAgICAgYW5hbHlzZXNbZiJkaWZmZXJlbmNlX2luX2Z1c2lvbl9nYWluX3ttZXRyaWN9Il0gPSB7CiAgICAgICAgICAgICJuX3N1YmplY3RzIjogbGVuKGRlbHRhKSwgImRpcnR5X2dhaW5fbWVhbiI6IGZsb2F0KGRpcnR5Lm1lYW4oKSksCiAgICAgICAgICAgICJjbGVhbl9nYWluX21lYW4iOiBmbG9hdChjbGVhbi5tZWFuKCkpLAogICAgICAgICAgICAiY2hhbmdlX21lYW4iOiBmbG9hdChkZWx0YS5tZWFuKCkpLAogICAgICAgICAgICAiYm9vdHN0cmFwXzk1Y2kiOiBbZmxvYXQoeCkgZm9yIHggaW4gbnAucXVhbnRpbGUoZHJhd3MsIFswLjAyNSwgMC45NzVdKV0sCiAgICAgICAgICAgICJ3aWxjb3hvbl9wIjogZmxvYXQod2lsY294b24oZGVsdGEpLnB2YWx1ZSkgaWYgbnAuYW55KGRlbHRhICE9IDApIGVsc2UgMS4wLAogICAgICAgIH0KCiAgICByZXN1bHQgPSB7InByb3RvY29sIjogInN1YmplY3QtZGlzam9pbnQgdmFsaWRhdGlvbjsgY2xlYW4gb3V0ZXIgdGVzdCBpbiBib3RoIGNvbmRpdGlvbnMiLAogICAgICAgICAgICAgICJzZWVkcyI6IGFyZ3Muc2VlZHMsICJhbmFseXNlcyI6IGFuYWx5c2VzfQogICAgb3V0cHV0ID0gYXJncy5vdXRfZGlyIC8gInBhaXJlZF9hbmFseXNpcy5qc29uIgogICAgb3V0cHV0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yKSkKICAgIHByaW50KGYiU2F2ZWQge291dHB1dH0iKQogICAgZm9yIG5hbWUsIHZhbHVlIGluIGFuYWx5c2VzLml0ZW1zKCk6CiAgICAgICAgZWZmZWN0ID0gdmFsdWUuZ2V0KCJtZWFuX3BhaXJlZF9kZWx0YSIsIHZhbHVlLmdldCgiY2hhbmdlX21lYW4iKSkKICAgICAgICBwcmludChmIntuYW1lfTogZGVsdGE9e2VmZmVjdDorLjRmfSwgOTUlIENJPXt2YWx1ZVsnYm9vdHN0cmFwXzk1Y2knXX0sICIKICAgICAgICAgICAgICBmInA9e3ZhbHVlWyd3aWxjb3hvbl9wJ106LjRnfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='))
assert hashlib.sha256((CODE_DIR / 'analyze_rigorous_loso.py').read_bytes()).hexdigest() == 'b20e65f1cc4e08797af7c6f8a10d7656c6a01b42c5e6b4e89724eda113ebfe4a'
(CODE_DIR / 'analyze_rigorous_group5.py').write_bytes(base64.b64decode('IiIiQW5hbHl6ZSBwYWlyZWQgZml2ZS1mb2xkIHJlc3VsdHMgd2l0aCBmb2xkLWNsdXN0ZXIgdW5jZXJ0YWludHkuCgpUaGUgZml2ZSBpbmRlcGVuZGVudGx5IHRyYWluZWQgb3V0ZXIgZm9sZHMsIHJhdGhlciB0aGFuIDQ5IHRlc3Qgc3ViamVjdHMsCmFyZSB0aGUgcmVzYW1wbGluZyB1bml0cy4gV2l0aCBvbmx5IGZpdmUgY2x1c3RlcnMsIGludGVydmFscyBhcmUgZXhwbG9yYXRvcnkuCiIiIgoKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwYW5kYXMgYXMgcGQKCmZyb20gYW5hbHl6ZV9yaWdvcm91c19sb3NvIGltcG9ydCBDT05ESVRJT05TLCBNRVRSSUNTLCBNT0RFTFMsIGxvYWRfY29tcGxldGUKClJPT1QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApERUZBVUxUX1JFU1VMVFMgPSBST09UIC8gIkJDSV9SZXNlYXJjaC9yZXN1bHRzL3JpZ29yb3VzX2dyb3VwNSIKCgpkZWYgc3VtbWFyaXplX2RlbHRhKGRlbHRhLCBmb2xkX2J5X3NpZCwgYm9vdHN0cmFwKToKICAgICIiIkF2ZXJhZ2UgcGFpcmVkIHN1YmplY3QgZWZmZWN0czsgcmVzYW1wbGUgd2hvbGUgb3V0ZXIgZm9sZHMgZm9yIENJLiIiIgogICAgaWYgbm90IGRlbHRhLmluZGV4LmVxdWFscyhmb2xkX2J5X3NpZC5pbmRleCk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiU3ViamVjdCBhbmQgZm9sZCBvcmRlciBkaWZmZXJzIikKICAgIGZvbGRfaWRzID0gc29ydGVkKGZvbGRfYnlfc2lkLnVuaXF1ZSgpKQogICAgaWYgZm9sZF9pZHMgIT0gWzEsIDIsIDMsIDQsIDVdOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBmaXZlIG91dGVyIGZvbGRzLCBmb3VuZCB7Zm9sZF9pZHN9IikKICAgIHZhbHVlcyA9IFtkZWx0YVtmb2xkX2J5X3NpZCA9PSBmb2xkXS50b19udW1weShkdHlwZT1mbG9hdCkgZm9yIGZvbGQgaW4gZm9sZF9pZHNdCiAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNDIpCiAgICBkcmF3cyA9IG5wLmVtcHR5KGJvb3RzdHJhcCwgZHR5cGU9ZmxvYXQpCiAgICBmb3IgaSBpbiByYW5nZShib290c3RyYXApOgogICAgICAgIGNob3NlbiA9IHJuZy5pbnRlZ2VycygwLCA1LCBzaXplPTUpCiAgICAgICAgZHJhd3NbaV0gPSBucC5jb25jYXRlbmF0ZShbdmFsdWVzW2pdIGZvciBqIGluIGNob3Nlbl0pLm1lYW4oKQogICAgcmV0dXJuIHsKICAgICAgICAibl9zdWJqZWN0cyI6IGxlbihkZWx0YSksICJuX291dGVyX2ZvbGRzIjogNSwKICAgICAgICAibWVhbl9wYWlyZWRfZGVsdGEiOiBmbG9hdChkZWx0YS5tZWFuKCkpLAogICAgICAgICJmb2xkX21lYW5fZGVsdGFzIjogW2Zsb2F0KHBhcnQubWVhbigpKSBmb3IgcGFydCBpbiB2YWx1ZXNdLAogICAgICAgICJmb2xkX2NsdXN0ZXJfYm9vdHN0cmFwXzk1Y2kiOiBbZmxvYXQoeCkgZm9yIHggaW4gbnAucXVhbnRpbGUoZHJhd3MsIFswLjAyNSwgMC45NzVdKV0sCiAgICAgICAgInBvc2l0aXZlX3N1YmplY3RzIjogaW50KChkZWx0YSA+IDApLnN1bSgpKSwKICAgICAgICAibmVnYXRpdmVfc3ViamVjdHMiOiBpbnQoKGRlbHRhIDwgMCkuc3VtKCkpLAogICAgfQoKCmRlZiBtYWluKCk6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1vdXRfZGlyIiwgdHlwZT1QYXRoLCBkZWZhdWx0PURFRkFVTFRfUkVTVUxUUykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tc2VlZHMiLCB0eXBlPWludCwgbmFyZ3M9IisiLCBkZWZhdWx0PVs0MiwgMTMzNywgMjAyNF0pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJvb3RzdHJhcCIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMDAwKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIGlmIGxlbihzZXQoYXJncy5zZWVkcykpICE9IGxlbihhcmdzLnNlZWRzKSBvciBhcmdzLmJvb3RzdHJhcCA8IDEwMDoKICAgICAgICBwYXJzZXIuZXJyb3IoIlNlZWRzIG11c3QgYmUgdW5pcXVlIGFuZCBib290c3RyYXAgbXVzdCBiZSBhdCBsZWFzdCAxMDAiKQogICAgZGYgPSBsb2FkX2NvbXBsZXRlKGFyZ3Mub3V0X2RpciwgYXJncy5zZWVkcykKICAgIGlmICJvdXRlcl9mb2xkIiBub3QgaW4gZGYuY29sdW1uczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJFeHBlY3RlZCBncm91cDUgcmVzdWx0cyB3aXRoIG91dGVyX2ZvbGQiKQogICAgaWYgZGYuZ3JvdXBieSgic2lkIikub3V0ZXJfZm9sZC5udW5pcXVlKCkubWF4KCkgIT0gMToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJPdXRlciBmb2xkcyBkaWZmZXIgYmV0d2VlbiBwYWlyZWQgcnVucyIpCiAgICBmb2xkcyA9IGRmLmdyb3VwYnkoInNpZCIpLm91dGVyX2ZvbGQuZmlyc3QoKS5zb3J0X2luZGV4KCkKICAgIGlmIHNvcnRlZChkZi5vdXRlcl9mb2xkLnVuaXF1ZSgpKSAhPSBbMSwgMiwgMywgNCwgNV06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiSW5jb21wbGV0ZSBvdXRlciBmb2xkcyIpCgogICAgcGVyX3N1YmplY3QgPSBkZi5ncm91cGJ5KFsiY29uZGl0aW9uIiwgIm1vZGVsX3R5cGUiLCAic2lkIl0pW2xpc3QoTUVUUklDUyldLm1lYW4oKQogICAgYW5hbHlzZXMgPSB7fQogICAgZm9yIG1ldHJpYyBpbiBNRVRSSUNTOgogICAgICAgIGRlZiBwYWlyKGEsIGIpOgogICAgICAgICAgICBmaXJzdCA9IHBlcl9zdWJqZWN0LnhzKGEpW21ldHJpY10uc29ydF9pbmRleCgpCiAgICAgICAgICAgIHNlY29uZCA9IHBlcl9zdWJqZWN0LnhzKGIpW21ldHJpY10uc29ydF9pbmRleCgpCiAgICAgICAgICAgIGlmIG5vdCBmaXJzdC5pbmRleC5lcXVhbHMoc2Vjb25kLmluZGV4KToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlVucGFpcmVkIHRlc3Qgc3ViamVjdHMiKQogICAgICAgICAgICByZXR1cm4gc3VtbWFyaXplX2RlbHRhKGZpcnN0IC0gc2Vjb25kLCBmb2xkcywgYXJncy5ib290c3RyYXApCgogICAgICAgIGZvciBtb2RlbCBpbiBNT0RFTFM6CiAgICAgICAgICAgIGFuYWx5c2VzW2YidHJhaW5pbmdfY2xlYW51cF97bW9kZWx9X3ttZXRyaWN9Il0gPSBwYWlyKAogICAgICAgICAgICAgICAgKCJjbGVhbl90cmFpbiIsIG1vZGVsKSwgKCJkaXJ0eV90cmFpbiIsIG1vZGVsKSkKICAgICAgICBmb3IgY29uZGl0aW9uIGluIENPTkRJVElPTlM6CiAgICAgICAgICAgIGFuYWx5c2VzW2YiZnVzaW9uX2dhaW5fe2NvbmRpdGlvbn1fe21ldHJpY30iXSA9IHBhaXIoCiAgICAgICAgICAgICAgICAoY29uZGl0aW9uLCAiZnVzaW9uIiksIChjb25kaXRpb24sICJlZWdfb25seSIpKQogICAgICAgIGNsZWFuX2dhaW4gPSAocGVyX3N1YmplY3QueHMoKCJjbGVhbl90cmFpbiIsICJmdXNpb24iKSlbbWV0cmljXQogICAgICAgICAgICAgICAgICAgICAgLSBwZXJfc3ViamVjdC54cygoImNsZWFuX3RyYWluIiwgImVlZ19vbmx5IikpW21ldHJpY10pLnNvcnRfaW5kZXgoKQogICAgICAgIGRpcnR5X2dhaW4gPSAocGVyX3N1YmplY3QueHMoKCJkaXJ0eV90cmFpbiIsICJmdXNpb24iKSlbbWV0cmljXQogICAgICAgICAgICAgICAgICAgICAgLSBwZXJfc3ViamVjdC54cygoImRpcnR5X3RyYWluIiwgImVlZ19vbmx5IikpW21ldHJpY10pLnNvcnRfaW5kZXgoKQogICAgICAgIGFuYWx5c2VzW2YiZGlmZmVyZW5jZV9pbl9mdXNpb25fZ2Fpbl97bWV0cmljfSJdID0gc3VtbWFyaXplX2RlbHRhKAogICAgICAgICAgICBjbGVhbl9nYWluIC0gZGlydHlfZ2FpbiwgZm9sZHMsIGFyZ3MuYm9vdHN0cmFwKQoKICAgIHJlc3VsdCA9IHsKICAgICAgICAicHJvdG9jb2wiOiAiZml4ZWQgZml2ZS1mb2xkIHN1YmplY3QtZGlzam9pbnQgZXZhbHVhdGlvbjsgY2xlYW4gdGVzdCBpbiBib3RoIGNvbmRpdGlvbnMiLAogICAgICAgICJzZWVkcyI6IGFyZ3Muc2VlZHMsCiAgICAgICAgInVuY2VydGFpbnR5IjogIlJlc2FtcGxlIGZpdmUgb3V0ZXIgZm9sZHMuIEV4cGxvcmF0b3J5IHdpdGggb25seSBmaXZlIGNsdXN0ZXJzOyBkbyBub3QgdHJlYXQgNDkgdGVzdCBzdWJqZWN0cyBhcyBpbmRlcGVuZGVudCBmaXR0ZWQgbW9kZWxzLiIsCiAgICAgICAgImFuYWx5c2VzIjogYW5hbHlzZXMsCiAgICB9CiAgICBvdXRwdXQgPSBhcmdzLm91dF9kaXIgLyAicGFpcmVkX2FuYWx5c2lzX2dyb3VwNS5qc29uIgogICAgb3V0cHV0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yKSkKICAgIHByaW50KGYiU2F2ZWQge291dHB1dH0iKQogICAgZm9yIG5hbWUsIHZhbHVlIGluIGFuYWx5c2VzLml0ZW1zKCk6CiAgICAgICAgcHJpbnQoZiJ7bmFtZX06IGRlbHRhPXt2YWx1ZVsnbWVhbl9wYWlyZWRfZGVsdGEnXTorLjRmfSwgIgogICAgICAgICAgICAgIGYiZm9sZC1jbHVzdGVyIDk1JSBDST17dmFsdWVbJ2ZvbGRfY2x1c3Rlcl9ib290c3RyYXBfOTVjaSddfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='))
assert hashlib.sha256((CODE_DIR / 'analyze_rigorous_group5.py').read_bytes()).hexdigest() == '271cda7f45b8e81040371382d5645d322113ed0f404e1f680276fcbdd1950b5e'
(CODE_DIR / 'excluded_epochs.json').write_bytes(base64.b64decode('...'))
assert hashlib.sha256((CODE_DIR / 'excluded_epochs.json').read_bytes()).hexdigest() == 'c59d37613123ae1408c9d9b1cd260f0a6a665aeb335c4051ab024955c2487075'
print("Verified code and exclusions:", CODE_DIR)


In [ ]:
# 3. 입력 검증 및 HDF5를 로컬 런타임 디스크에 복사
import json, shutil, subprocess, sys, time
manifest = json.loads((CODE_DIR / 'excluded_epochs.json').read_text())
assert len(manifest) == 51 and sum(len(v['excl_idx']) for v in manifest.values()) == 753
source = DRIVE_ROOT / 'preprocessed/member_A'
assert source.is_dir(), f'HDF5 폴더가 없습니다: {source}'
needed = [source / f'sub-{int(s):02d}_member_A.h5' for s in sorted(manifest, key=int)]
missing = [p.name for p in needed if not p.is_file()]
assert not missing, f'HDF5 파일 누락: {missing}'
bytes_needed = sum(p.stat().st_size for p in needed)
local = Path('/content/bci_data')
local.mkdir(exist_ok=True)
free = shutil.disk_usage('/content').free
remaining_bytes = sum(p.stat().st_size for p in needed if not (local / p.name).is_file() or (local / p.name).stat().st_size != p.stat().st_size)
assert free >= remaining_bytes + 4_000_000_000, f'로컬 디스크 부족: 추가 필요 {remaining_bytes/1e9:.1f}GB, 여유 {free/1e9:.1f}GB'
for i, p in enumerate(needed, 1):
    target = local / p.name
    if not target.is_file() or target.stat().st_size != p.stat().st_size:
        shutil.copy2(p, target)
    if i % 10 == 0 or i == len(needed):
        print(f'{i}/{len(needed)} 파일 확인')
import h5py
for sid in sorted(manifest, key=int):
    with h5py.File(local / f'sub-{int(sid):02d}_member_A.h5') as f:
        n = manifest[sid]['n_total']
        assert f['labels'].shape[0] == n, f'라벨 수 불일치: s{int(sid):02d}'
        assert f['eeg/epochs'].shape == (n, 64, 2304), f'EEG shape 불일치: s{int(sid):02d}'
        assert f['emg/epochs'].shape == (n, 4, 2304), f'sEMG shape 불일치: s{int(sid):02d}'
OUT_DIR = DRIVE_ROOT / 'results/rigorous_group5'
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = local
EXCL_JSON = CODE_DIR / 'excluded_epochs.json'
print('준비 완료:', len(needed), 'subjects; output:', OUT_DIR)

# Stream child stdout/stderr into the notebook and retain the actual error.
from collections import deque
def run_training(cmd):
    tail = deque(maxlen=40)
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line.rstrip())
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Training process exited with code {return_code}. Last output:\n' + '\n'.join(tail))


## 재학습

각 실행은 시작 시 `완료 N/5 folds`, fold 시작·완료 시 `[N/5]`, 학습 중 5 epoch마다 검증 F1을 출력합니다. 바로 아래 진행 현황 셀은 18개 조합의 완료 fold 수를 보여줍니다. 중단되면 준비 셀 1–3을 다시 실행하고 같은 학습 셀을 다시 실행하세요. **한 fold의 테스트 피험자 점수는 모두 계산된 뒤에만 CSV에 저장됩니다.**


In [ ]:
# 저장된 18개 조합의 진행 상태. 언제든 다시 실행할 수 있습니다.
import csv
def show_progress():
    print(f'{"seed":>5}  {"model":<8}  {"condition":<11}  {"folds":>7}  {"subjects":>9}')
    total = 0
    for seed in (42, 1337, 2024):
        for model in ('eeg_only', 'emg_only', 'fusion'):
            for condition in ('dirty_train', 'clean_train'):
                path = OUT_DIR / f'{condition}_{model}_seed{seed}.csv'
                if path.is_file():
                    with path.open(newline='') as stream:
                        rows = list(csv.DictReader(stream))
                    folds = len({int(row['outer_fold']) for row in rows})
                    subjects = len({int(row['sid']) for row in rows})
                else:
                    folds = subjects = 0
                total += folds
                print(f'{seed:>5}  {model:<8}  {condition:<11}  {folds:>2}/5  {subjects:>2}/49')
    print(f'Total: {total}/90 model fits complete')
show_progress()


## 선택: 첫 fold로 소요시간 측정

아래 셀은 fusion / dirty_train / seed 42의 첫 fold **1회만 학습**합니다. 출력되는 `elapsed`는 이 조합의 실제 측정값입니다. 나중에 전체 fusion 실행 셀을 누르면 완료된 첫 fold를 건너뜁니다. GPU와 Colab 부하에 따라 모델별 시간은 다를 수 있습니다. 이 셀을 실행하지 않아도 됩니다.


In [ ]:
# 선택적 시간 측정: 전체 90회 중 1회만 수행하고 저장합니다.
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '42',
       '--folds', '1', '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=eeg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'eeg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=eeg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'eeg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=emg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'emg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=emg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'emg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=fusion, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=fusion, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'fusion', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=eeg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'eeg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=eeg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'eeg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=emg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'emg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=emg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'emg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=fusion, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=fusion, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'fusion', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=eeg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'eeg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=eeg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'eeg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=emg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'emg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=emg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'emg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=fusion, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=fusion, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_group5.py'),
       '--condition', 'clean_train', '--model_type', 'fusion', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


## 모든 조합 완료 후 분석

18개 CSV가 각각 49명의 피험자를 포함하는지 확인합니다. 5개 외부 fold를 묶어 효과와 fold 단위 부트스트랩 구간을 산출합니다. 이 구간은 fold가 5개라 탐색적으로 해석하세요.


In [ ]:
subprocess.run([sys.executable, str(CODE_DIR / 'analyze_rigorous_group5.py'),
                '--out_dir', str(OUT_DIR), '--seeds', '42', '1337', '2024'], check=True)
print('분석 결과:', OUT_DIR / 'paired_analysis_group5.json')
